# **Amygdala stimulus responses — SocialPresentation (GRIN)**
### *population PSTHs, single-neuron tuning, food vs social selectivity, FOV map*

Works from the presentation-locked tensor written by `processing_main_abf.ipynb`
(`tensor_dff_raw.npy` — not baseline-corrected —, `tensor_labels.npy`, `tensor_meta.npy`, `trials_info.csv`).

**Baseline logic.** Every trial is baseline-corrected to −2 to 0 s (the flat stretch just before
onset; activity falls from ~−6 s to ~−1.5 s after an inter-trial event), for traces **and** statistics.
The per-trial correction is required: raw activity falls steadily across each block and decays after
each acquisition starts, so with the fixed order raw levels differ by block position (Section 3b).
"Responsive" means different from the **empty cage**, since Empty itself drives a post-onset rise.
Code lives in `amygdala_analysis.py` (stimulus table, categories and contrasts are defined at its top).

- **Section 0** — config (all analysis choices fixed here) + load
- **Section 1** — population PSTHs (per stimulus, per category) + neuron heatmaps
- **Section 2** — single neurons: trial-averaged traces + single-trial rasters
- **Section 3** — selectivity: responsiveness, tuning, category AUCs, neuron classes
- **Section 3b** — controls for the fixed stimulus order
- **Section 4** — suite2p mean image with ROIs coloured by preference

> **Design caveat.** Every block presents the stimuli in the same order
> (Empty → FamFood → NewFood → NewObject → Male1 → Female1 → Male2 → Female2), so
> stimulus identity is confounded with position in the block. Section 3b tests pieces
> of this (Male1 vs Male2, split-half, drift, baseline carry-over) but cannot remove it.

**Plot markers:** grey shading = response window · dotted = median entry TTL ·
black dashed = presentation onset · grey dashed = median presentation offset.

## **Section 0 : Config + load**

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
from neural_analysis import set_style                      # shared figure style
import amygdala_analysis as aa
from amygdala_analysis import StimSession, CATEGORIES, save_fig
set_style(font="Arial", size=7)

# ---- WHICH SESSION (the only block you normally edit) ----
ROOT     = "/Volumes/Akhil Data/Naomi data"
MOUSE    = "GRINPL1"
DATE     = "2026-09-03"
SIGNAL   = "dff"               # "zdff" | "dff" | "deconv"

DATA     = os.path.join(ROOT, "ProcessedData", MOUSE, DATE)     # written by processing_main_abf.ipynb
SUITE2P  = os.path.join(ROOT, "Suite2p", DATE, "plane0")

# ---- ANALYSIS CHOICES (fix these before looking at results) ----
BASELINE_WIN   = (-2.0, 0.0)   # s rel. onset — baseline for TRACES (flat pre-onset stretch)
RESPONSE_WIN   = (0.0, 8.0)    # s rel. onset; e.g. (0, 8) for the full presentation
TRIAL_BASELINE = True          # stats on window − per-trial baseline (keep True; False = raw levels, confounded by block position)
RESP_REF       = "Empty"       # responsiveness reference: "Empty" (block-paired) or "baseline"
ALPHA        = 0.05             # FDR level (BH)
N_SHUFFLE    = 1000             # permutations for AUC p-values
WITHIN_BLOCK = True             # permute labels within blocks (controls for slow drift)
SEED         = 0

# ---- DISPLAY ----
PLOT_WIN  = (-2.0, 8.0)         # s rel. onset shown on all time-axis plots (display only)
SMOOTH    = 2                   # gaussian sigma (frames) — display only, never used for stats
FOV_IMAGE = "meanImg"           # "meanImg" | "meanImgE" | "max_proj" | "Vcorr"
OUT_DIR   = os.path.join(DATA, "figures")   # figures + tables for this mouse/session (None = don't save)

In [ ]:
ss = StimSession.load(DATA, signal=SIGNAL).set_windows(baseline=BASELINE_WIN, response=RESPONSE_WIN,
                                                       trial_baseline=TRIAL_BASELINE)
ss.set_plot_window(PLOT_WIN)
ss.load_fov(SUITE2P)
print(f"tensor {ss.X.shape}  (trials, neurons, frames) | fs = {ss.fs:g} Hz | onset at frame {ss.pre}")
print(f"trace baseline {ss.baseline_win} s | response {ss.response_win} s | "
      f"per-trial baseline in stats: {ss.trial_baseline} | responsiveness vs {RESP_REF}")
print(f"session {ss.meta.get('mouse')} {ss.meta.get('date')} | median entry "
      f"{'n/a' if ss.entry_t is None else f'{ss.entry_t:.2f} s'}, median hold {ss.offset_t:.2f} s")
print(f"trials usable for stats: {ss.valid.sum()}/{ss.n_trials}")
pd.DataFrame({"trials": [int(ss.trials_of(s).sum()) for s in ss.stims],
              "category": [next(c for c, v in CATEGORIES.items() if s in v) for s in ss.stims]},
             index=ss.stims).T

## **Section 1 : Population PSTHs**

Each neuron's trial-averaged trace (baseline-corrected to `BASELINE_WIN`), then mean ± SEM **across the 16 neurons**
(legend shows the number of trials per stimulus).

In [ ]:
fig = ss.plot_population_psth(smooth_sigma=SMOOTH, figsize=(3.2, 2), dpi=250)
save_fig(fig, "psth_by_stimulus", OUT_DIR); plt.show()

In [ ]:
fig = ss.plot_population_psth(groups=["control", "food", "object", "social"],
                              smooth_sigma=SMOOTH, figsize=(3.2, 2), dpi=250)
save_fig(fig, "psth_by_category", OUT_DIR); plt.show()

Neuron × time heatmap per stimulus. All panels share one neuron order (sorted by
response-window activity to `sort_by`: a stimulus, a category, or `"all"`).

In [ ]:
fig = ss.plot_heatmaps(sort_by="all", smooth_sigma=1, dpi=250)
save_fig(fig, "heatmaps_by_stimulus", OUT_DIR); plt.show()

## **Section 2 : Single neurons**

Trial-averaged trace per stimulus for every neuron (± SEM across trials; y-axes independent).

In [ ]:
fig = ss.plot_neuron_grid(smooth_sigma=SMOOTH, dpi=250)
save_fig(fig, "neurons_by_stimulus", OUT_DIR); plt.show()

In [ ]:
fig = ss.plot_neuron_grid(groups=["control", "food", "object", "social"], smooth_sigma=SMOOTH, dpi=250)
save_fig(fig, "neurons_by_category", OUT_DIR); plt.show()

Single-trial rasters (trials grouped by stimulus, session order within each group) — shows how reliable each average is.

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(9, 11), dpi=200)
for n, ax in enumerate(axs.flat):
    ss.plot_neuron_raster(n, smooth_sigma=1, ax=ax)
    if n % 4:
        ax.set_yticklabels([])
fig.tight_layout()
save_fig(fig, "neuron_rasters", OUT_DIR); plt.show()

## **Section 3 : Selectivity**

Response per trial = mean ΔF/F in `RESPONSE_WIN` minus that trial's mean in `BASELINE_WIN`.

* **Responsiveness** — each stimulus vs the empty cage, paired by block (per-block means, Wilcoxon
  signed-rank), BH-FDR over all 16 × 7 tests. `RESP_REF="baseline"` gives the old window-vs-baseline test.
* **Tuning** — Kruskal–Wallis across the 8 stimuli; preferred stimulus/category and lifetime sparseness
  (0 = equal drive to all, 1 = one stimulus) on responses relative to Empty. The tuning matrix and
  category scatter also show responses **minus Empty**, so each neuron's offset cancels.
* **Category contrasts** — ROC AUC per neuron (A vs B trials; > 0.5 = more activity to A),
  permutation p (labels shuffled within block), BH-FDR across neurons per contrast. Contrasts are in `aa.CONTRASTS`.
* **Class** — `social` / `food` / `object` if that category's *vs rest* AUC is > 0.5 and significant
  (highest AUC wins if several); otherwise `non-selective` if responsive to any stimulus, else `non-responsive`.
  Preference is relative — "less suppressed" also counts — so check the sign in the tuning matrix.

In [ ]:
classes = ss.classify(alpha=ALPHA, n_shuffle=N_SHUFFLE, within_block=WITHIN_BLOCK, seed=SEED,
                      reference=RESP_REF)
print(classes["class"].value_counts().to_string())
classes[["neuron", "class", "responsive", "pref_stim", "pref_category", "sparseness", "kw_q",
         "auc_social_vs_food", "q_social_vs_food", "auc_social_vs_rest", "auc_food_vs_rest", "auc_object_vs_rest"]].round(3)

In [ ]:
fig = ss.plot_tuning_matrix(alpha=ALPHA, dpi=250)
save_fig(fig, "tuning_matrix", OUT_DIR); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(7, 2.2), dpi=250, gridspec_kw={"width_ratios": [1, 1, 0.8]})
ss.plot_category_scatter("food", "social", ax=axs[0])
ss.plot_category_scatter("object", "social", ax=axs[1])
ss.plot_class_counts(ax=axs[2])
fig.tight_layout()
save_fig(fig, "category_scatter_and_counts", OUT_DIR); plt.show()

Per-neuron AUC for every contrast (filled = FDR-significant).

In [ ]:
fig = ss.plot_auc_summary(figsize=(4, 2), dpi=250)
save_fig(fig, "auc_summary", OUT_DIR); plt.show()

auc = ss.contrast_auc(n_shuffle=N_SHUFFLE, within_block=WITHIN_BLOCK, seed=SEED, alpha=ALPHA)
print("significant neurons per contrast:")
print(auc.groupby("contrast", sort=False)["sig"].sum().to_string())
auc[auc.sig].sort_values(["contrast", "auc"]).round(3)

Examples: traces of the neurons in each preference class.

In [ ]:
for cls in ["social", "food", "object"]:
    ids = classes.loc[classes["class"] == cls, "neuron"].tolist()
    if not ids:
        print(f"no {cls}-preferring neurons"); continue
    fig = ss.plot_neuron_grid(neurons=ids, ncols=min(4, len(ids)), smooth_sigma=SMOOTH, dpi=250,
                              titles={n: cls for n in ids})
    save_fig(fig, f"examples_{cls}", OUT_DIR); plt.show()

## **Section 3b : Controls for the fixed stimulus order**

**Same category, different animal and block position.** If a neuron's preference is
about the category (and not identity or position in the block), its responses to
male 1 and male 2 (and female 1 and female 2) should agree — points near the unity line.

In [ ]:
ident, fig = ss.identity_consistency()
save_fig(fig, "control_identity", OUT_DIR); plt.show()
ident.round(3)

**Split-half reliability** (odd vs even blocks): per-neuron correlation of the 8-stimulus
tuning vector, and whether the social − food difference replicates across halves.

In [ ]:
split, fig = ss.split_half()
save_fig(fig, "control_split_half", OUT_DIR); plt.show()
split.round(3)

**Drift across the session** — Spearman ρ of each neuron's stimulus-centred response vs
trial order. **Raw baseline by block position** — the un-subtracted baseline level for each
stimulus, i.e. each position in the block (Friedman test, blocks as repeated measures). A trend
here is why stats use the per-trial baseline; the blocks × position table shows the
start-of-acquisition transient (first blocks of each t-series).

In [ ]:
drift = ss.drift()
print("neurons with significant drift (q < ALPHA):", drift.loc[drift.q < ALPHA, "neuron"].tolist())
carry, fig, base_tab = ss.baseline_carryover()
save_fig(fig, "control_baseline_by_position", OUT_DIR); plt.show()
print("neurons whose raw baseline differs by block position (Friedman q < ALPHA):",
      carry.loc[carry.friedman_q < ALPHA, "neuron"].tolist())
display(base_tab.round(2))          # population-mean raw baseline, blocks x position
drift.merge(carry, on="neuron").round(3)

## **Section 4 : FOV map**

Suite2p image with each neuron's footprint. **Left:** filled by class (non-responsive = white outline).
**Right:** filled by social-vs-food AUC (red = social, blue = food); dashed outline = not significant.
Change `contrast=` to map any contrast in `aa.CONTRASTS`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3.6), dpi=250)
ss.plot_fov(color_by="class", image=FOV_IMAGE, ax=axs[0])
ss.plot_fov(color_by="auc", contrast="social_vs_food", image=FOV_IMAGE, ax=axs[1])
fig.tight_layout()
save_fig(fig, "fov_preference", OUT_DIR); plt.show()

Whole FOV (uncropped) for orientation.

In [ ]:
fig = ss.plot_fov(color_by="class", image=FOV_IMAGE, crop=False, figsize=(4, 4), dpi=200)
plt.show()

## **Save results**

In [ ]:
if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)
    classes.to_csv(os.path.join(OUT_DIR, "neuron_classes.csv"), index=False)
    auc.to_csv(os.path.join(OUT_DIR, "contrast_auc.csv"), index=False)
    ss.responsiveness(ALPHA, reference=RESP_REF).to_csv(os.path.join(OUT_DIR, "responsiveness.csv"), index=False)
    print("saved to", OUT_DIR)
else:
    print("OUT_DIR is None — nothing saved")